# Step 6 (Baseline) — RUL BiLSTM, no augmentation, no multi-task

**RESS 2025 — GAN-Conformal-RUL**

Establishes the **anchor RMSE**: a 2-layer BiLSTM with temporal attention and a single RUL
regression head, trained on real data only. Every later contribution — GAN augmentation, the
stage-classification head, RUL-loss masking — is measured against this number.

| | |
|---|---|
| **Encoder** | 2-layer BiLSTM (hidden 64/direction) + temporal attention |
| **Head** | RUL regression → sigmoid → [0,1] |
| **Loss** | MSE on normalised RUL |
| **Trained on** | real training windows only |
| **Reported** | RMSE & MAE in **minutes** (primary), normalised RMSE (secondary) |

Minutes are recovered by multiplying each window's normalised RUL by its bearing's lifetime —
the unit used across the XJTU-SY literature (Lu et al. 2022 cumulative RMSE 21.90 min is the
target to beat).

## 1. Setup

In [1]:
import os, sys, shutil
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = f'{REPO_PATH}/figures'; os.makedirs(SAVE_DIR, exist_ok=True)
import torch
print('CUDA:', torch.cuda.is_available())

Cloning into '/content/RESS_2025_GAN_Conformal_RUL'...
remote: Enumerating objects: 330, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 330 (delta 83), reused 5 (delta 5), pack-reused 201 (from 1)
Receiving objects: 100% (330/330), 7.96 MiB | 19.54 MiB/s, done.
Resolving deltas: 100% (176/176), done.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUDA: True


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import prepare_all_folds, build_folds, WINDOW_SIZE
from model import (ModelConfig, RULTrainer, rul_metrics,
                   per_bearing_rmse, cumulative_rmse, phm_score)

[health_indicator] v2.0-guarded-fpt loaded  (FPT: 5 consecutive x max(mu+3sigma, mu*1.10))


## 2. Rebuild data (Steps 2–3, log_clip scaling)

In [3]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_ROOT, f'not found: {CANDIDATES}'

all_data = XJTUSYLoader(DATA_ROOT).load_all()
pipeline = HealthIndicatorPipeline(fpt_consecutive=5, fpt_min_relative_rise=0.20)
results = pipeline.process_all(all_data, verbose=False)

fold_data = prepare_all_folds(results, scaling_method='log_clip', verbose=False)
folds = build_folds(results)
print(f'{len(fold_data)} folds ready.')


=== Condition 1: 35.0 Hz, 12.0 kN ===
Loading Bearing1_1 (1-1): 123 recordings, failure mode: Outer race
  -> Shape: (123, 32768, 2), Memory: 32.2 MB
Loading Bearing1_2 (1-2): 161 recordings, failure mode: Outer race
  -> Shape: (161, 32768, 2), Memory: 42.2 MB
Loading Bearing1_3 (1-3): 158 recordings, failure mode: Outer race
  -> Shape: (158, 32768, 2), Memory: 41.4 MB
Loading Bearing1_4 (1-4): 122 recordings, failure mode: Cage
  -> Shape: (122, 32768, 2), Memory: 32.0 MB
Loading Bearing1_5 (1-5): 52 recordings, failure mode: Outer race + Ball
  -> Shape: (52, 32768, 2), Memory: 13.6 MB

=== Condition 2: 37.5 Hz, 11.0 kN ===
Loading Bearing2_1 (2-1): 491 recordings, failure mode: Inner race
  -> Shape: (491, 32768, 2), Memory: 128.7 MB
Loading Bearing2_2 (2-2): 161 recordings, failure mode: Outer race
  -> Shape: (161, 32768, 2), Memory: 42.2 MB
Loading Bearing2_3 (2-3): 533 recordings, failure mode: Cage
  -> Shape: (533, 32768, 2), Memory: 139.7 MB
Loading Bearing2_4 (2-4): 42 re

## 3. Bearing lifetimes for the minutes conversion

Each test window's normalised RUL is converted to minutes by multiplying by its bearing's
total lifetime. We build a per-window lifetime vector for each fold's test set, in the same
window order that `prepare_fold` produced (bearings concatenated in the order listed in the
fold's `test` field).

In [4]:
# Per-window metadata for the test set of a fold, matching prepare_fold's
# concatenation order (bearings in the order listed in fold['test']).
def test_meta(fold, results, window_size=WINDOW_SIZE):
    """Returns per-window (lifetime_min, bearing_id) arrays and a dict of
    prognostic durations dT = t_EOL - t_FPT + 1 per bearing."""
    life, bids = [], []
    dT = {}
    for bid in fold['test']:
        r = results[bid]
        n_rec = r['features'].shape[0]
        n_win = max(0, n_rec - window_size + 1)
        life.extend([n_rec] * n_win)
        bids.extend([bid] * n_win)
        # prognostic duration: FPT to EOL. results carries fpt index per bearing.
        fpt = r.get('fpt', 0)
        dT[bid] = n_rec - fpt + 1
    return np.array(life, float), np.array(bids), dT

# sanity: metadata length must equal test window count
for d, f in zip(fold_data, folds):
    life, bids, _ = test_meta(f, results)
    assert len(life) == len(d['X_test']) == len(bids), \
        f"fold {f['fold']}: {len(life)} meta vs {len(d['X_test'])} windows"
print('per-window metadata aligned with test windows for all folds.')

per-window metadata aligned with test windows for all folds.


## 4. Train the baseline, one fold at a time

Train on real training windows, early-stop on validation RMSE, evaluate on the held-out test
bearings. RUL target only — the stage labels are ignored for the baseline.

In [5]:
cfg = ModelConfig(epochs=100, patience=15, lr=1e-3)

rows = []
per_bearing_all = {}
trainers = {}
for d, f in zip(fold_data, folds):
    k = d['fold']
    print(f"\n{'='*56}\nFOLD {k}\n{'='*56}")
    tr = RULTrainer(cfg)
    tr.fit(d['X_train'], d['y_rul_train'],
           d['X_val'],   d['y_rul_val'],
           stage_train=d['y_stage_train'], stage_val=d['y_stage_val'],
           mask_healthy=True, verbose=True)

    trainers[k] = tr

    y_pred = tr.predict(d['X_test'])
    life, bids, dT = test_meta(f, results)
    stages_1idx = d['y_stage_test'] + 1

    m = rul_metrics(d['y_rul_test'], y_pred, life)
    pb = per_bearing_rmse(d['y_rul_test'], y_pred, bids, life,
                          stages=stages_1idx, post_fpt_only=True)
    per_bearing_all[k] = {b: v for b, v in pb.items() if b != '_mean'}
    cum = cumulative_rmse(pb, dT)
    sc = phm_score(d['y_rul_test'], y_pred, bids)

    m.update({'fold': k, 'per_bearing_mean': pb.get('_mean', float('nan')),
              'cumulative': cum, 'phm_score': sc.get('_score', float('nan'))})
    rows.append(m)
    print(f"  TEST  pooled-RMSE {m['rmse_min']:.2f} min | "
          f"per-bearing {pb.get('_mean', float('nan')):.2f} | "
          f"cumulative {cum:.2f} | norm {m['rmse_norm']:.4f} | "
          f"score {sc.get('_score', float('nan')):.3f}")


FOLD 1
  RUL masking: 2100/3560 post-FPT windows kept for regression
  epoch   0 | train RMSE 0.1737 | val RMSE 0.2502 | best 0.2502
  epoch  10 | train RMSE 0.0332 | val RMSE 0.2844 | best 0.2502
  early stop at epoch 15 (no val improvement for 15)
  TEST  pooled-RMSE 621.49 min | per-bearing 313.51 | cumulative 642.47 | norm 0.2643 | score 0.000

FOLD 2
  RUL masking: 545/4876 post-FPT windows kept for regression
  epoch   0 | train RMSE 0.3575 | val RMSE 0.3193 | best 0.3193
  epoch  10 | train RMSE 0.0598 | val RMSE 0.4819 | best 0.3193
  early stop at epoch 15 (no val improvement for 15)
  TEST  pooled-RMSE 830.78 min | per-bearing 131.06 | cumulative 291.49 | norm 0.3401 | score 0.000

FOLD 3
  RUL masking: 572/4879 post-FPT windows kept for regression
  epoch   0 | train RMSE 0.3492 | val RMSE 0.3056 | best 0.3056
  epoch  10 | train RMSE 0.0538 | val RMSE 0.4979 | best 0.3056
  early stop at epoch 15 (no val improvement for 15)
  TEST  pooled-RMSE 107.86 min | per-bearing 75.7

In [7]:
import numpy as np
d = fold_data[1]  # fold 1
tr = trainers[1]
yp = tr.predict(d['X_test'])
yt = d['y_rul_test']
st = d['y_stage_test']  # 0,1,2

for s, name in [(0,'healthy'),(1,'early'),(2,'near-failure')]:
    m = st == s
    if m.sum():
        rmse = np.sqrt(np.mean((yp[m]-yt[m])**2))
        print(f"{name:12s}: n={m.sum():5d}  RMSE {rmse:.4f}  "
              f"pred_mean {yp[m].mean():.3f}  true_mean {yt[m].mean():.3f}")

healthy     : n= 1223  RMSE 0.2793  pred_mean 0.503  true_mean 0.748
early       : n=  810  RMSE 0.1445  pred_mean 0.276  true_mean 0.396
near-failure: n=  692  RMSE 0.0718  pred_mean 0.146  true_mean 0.125


In [8]:
import numpy as np
for k in [1,2,3,4,5]:
    d = fold_data[k-1]; tr = trainers[k]
    yp = tr.predict(d['X_test']); yt = d['y_rul_test']; st = d['y_stage_test']
    post = st >= 1
    rmse_post = np.sqrt(np.mean((yp[post]-yt[post])**2))
    print(f"Fold {k}: post-FPT norm-RMSE {rmse_post:.4f}  (all-window {np.sqrt(np.mean((yp-yt)**2)):.4f})")

Fold 1: post-FPT norm-RMSE 0.2764  (all-window 0.2643)
Fold 2: post-FPT norm-RMSE 0.1467  (all-window 0.3401)
Fold 3: post-FPT norm-RMSE 0.1932  (all-window 0.2355)
Fold 4: post-FPT norm-RMSE 0.3064  (all-window 0.2658)
Fold 5: post-FPT norm-RMSE 0.3067  (all-window 0.4566)


## 5. Baseline results — the anchor

In [6]:
df = pd.DataFrame(rows)[['fold','cumulative','rmse_norm','mae_norm','r2','phm_score']]
df.columns = ['fold','cumul_min','norm_rmse','norm_mae','r2','phm_score']
print(df.to_string(index=False))
print('-'*60)
print(f"  MEAN cumulative (Lu-comparable): {df['cumulative_min'].mean():6.2f} min")
print(f"  MEAN normalised RMSE:            {df['norm_rmse'].mean():.4f}  (BiLSTM benchmark ~0.158)")
print(f"  MEAN normalised MAE:             {df['norm_mae'].mean():.4f}  (BiLSTM benchmark ~0.126)")
print(f"  MEAN R2:                         {df['r2'].mean():.4f}")
print(f"  MEAN PHM score (higher better):  {df['phm_score'].mean():.3f}")
print(f"\nLu et al. 2022 cumulative RMSE: 21.90 (HP-JT, their best) | 31.00 (HP, plain predictor)")
print(f"Compare our baseline cumulative to their PLAIN predictor (31.00) — the fair baseline match.")

 fold  cumul_min  norm_rmse  norm_mae        r2    phm_score
    1 642.472050   0.264317  0.227834  0.122772 2.132351e-30
    2 291.493292   0.340081  0.272304 -0.454076 1.669625e-48
    3  91.795284   0.235544  0.198901  0.216184 2.536086e-56
    4 340.395718   0.265842  0.229957  0.108893 2.134610e-39
    5  81.392128   0.456595  0.384684 -2.178884 1.747167e-01
------------------------------------------------------------


KeyError: 'cumulative_min'

In [ ]:
# Training curves — confirm clean convergence and that early stopping fired sensibly
fig, axes = plt.subplots(1, len(trainers), figsize=(4*len(trainers), 3), sharey=True)
if len(trainers) == 1: axes = [axes]
for ax, (k, tr) in zip(axes, trainers.items()):
    ax.plot(tr.history['train_rmse'], label='train', lw=1.2)
    ax.plot(tr.history['val_rmse'], label='val', lw=1.2)
    ax.set_title(f'Fold {k}'); ax.set_xlabel('epoch'); ax.grid(alpha=0.3)
    if k == list(trainers)[0]: ax.set_ylabel('normalised RMSE'); ax.legend(fontsize=8)
plt.suptitle('Baseline training curves', y=1.03)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/06_baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Next

This baseline is the anchor. The next iterations layer on, one at a time, each as an ablation row:

1. **+GAN** — load frozen generators from Drive, augment the training set with synthetic
   near-failure windows at `augment_ratio`, retrain, compare RMSE. *Tests C1.*
2. **+Multi-task** — add the stage-classification head and the combined loss. *Tests C2.*
3. **+RUL masking** — mask the RUL loss to post-FPT windows. *The deferred Step 3 decision.*

Keeping them incremental means any change in RMSE is attributable to exactly one thing.

In [ ]:
d = fold_data[0]
tr = trainers[1]  # if still in memory; else retrain fold 1
yp = tr.predict(d['X_test'])
yt = d['y_rul_test']
print(f"pred range: [{yp.min():.3f}, {yp.max():.3f}]  std {yp.std():.3f}")
print(f"true range: [{yt.min():.3f}, {yt.max():.3f}]  std {yt.std():.3f}")
print(f"pred mean {yp.mean():.3f} | true mean {yt.mean():.3f}")
